Retrieve places with Overpass and enrich them with the Google Places API.


In [ ]:
API_KEY = 'YOUR_GOOGLE_MAPS_API_KEY'
overpass_url = "https://overpass-api.de/api/interpreter"

Get accommodation from OpenStreetMap.


In [ ]:
# Fetch places and include room and bed counts where tagged.
def get_osm_data(tag_key, tag_value):
    query = f"""
    [out:json][timeout:25];
    area["name"="Pisa"]["boundary"="administrative"]["admin_level"="8"]->.searchArea;
    (
      node["{tag_key}"="{tag_value}"](area.searchArea);
      way["{tag_key}"="{tag_value}"](area.searchArea);
      relation["{tag_key}"="{tag_value}"](area.searchArea);
    );
    out center tags;
    """
    response = requests.get(overpass_url, params={'data': query})
    data = response.json()
    elements = data["elements"]

    places = []
    for el in elements:
        tags = el.get("tags", {})
        name = tags.get("name")
        lat = el.get("lat") or el.get("center", {}).get("lat")
        lon = el.get("lon") or el.get("center", {}).get("lon")
        rooms = tags.get("rooms")
        beds = tags.get("beds")
        if name and lat and lon:
            places.append({
                "name": name,
                "lat": lat,
                "lon": lon,
                "category": tag_value,
                "rooms": rooms,
                "beds": beds
            })
    return places

# Fetch accommodations
hotels = get_osm_data("tourism", "hotel")
hostels = get_osm_data("tourism", "hostel")
guesthouses = get_osm_data("tourism", "guest_house")
accommodations = hotels + hostels + guesthouses

# Create DataFrame
df_accommodations = pd.DataFrame(accommodations)

print(f"Found {len(df_accommodations)} places for accomodation from OpenStreetMap.")

Found 125 places for accomodation from OpenStreetMap.


In [ ]:
# Add Google ratings and place IDs to the OpenStreetMap rows.
def enrich_with_google(df):
    enriched = []
    for _, row in df.iterrows():
        query = f"{row['name']} {row['category']} Pisa Italy"
        lat, lon = row["lat"], row["lon"]
        url = (
            f"https://maps.googleapis.com/maps/api/place/textsearch/json?"
            f"query={requests.utils.quote(query)}&location={lat},{lon}&radius=100&key={API_KEY}"
        )
        try:
            resp = requests.get(url)
            data = resp.json()
            if data["status"] == "OK" and data["results"]:
                top = data["results"][0]
                enriched.append({
                    "name": row["name"],
                    "lat": lat,
                    "lon": lon,
                    "category": row["category"],
                    "rooms": row.get("rooms"),
                    "beds": row.get("beds"),
                    "google_name": top.get("name"),
                    "user_ratings_total": top.get("user_ratings_total"),
                    "rating": top.get("rating"),
                    "place_id": top.get("place_id")
                })
        except Exception as e:
            print(f"Error for {row['name']}: {e}")
        time.sleep(0.5)
    return pd.DataFrame(enriched)

# Run enrichment for a small sample for demo (full run can be done locally)
df_enriched_accomodation = enrich_with_google(df_accommodations)

df_enriched_accomodation.head()
print(f"{len(df_enriched_accomodation)} places for accomodation")

125 places for accomodation


Retrieve attractions from map grid cells with the Google Places API.


In [ ]:
# Bounding box for central Pisa.
lat_min, lat_max = 43.6620, 43.7400
lon_min, lon_max = 10.3280, 10.4550

# Step size of about 100 metres.
step = 0.001

# Generate the grid points.
lat_points = np.arange(lat_min, lat_max, step)
lon_points = np.arange(lon_min, lon_max, step)
grid_points = [(round(lat, 5), round(lon, 5)) for lat in lat_points for lon in lon_points]

# Write the grid points to CSV.
df_points = pd.DataFrame(grid_points, columns=["latitude", "longitude"])
df_points.to_csv("pisa_grid_100m.csv", index=False)

print(f"Generated {len(grid_points)} points")


In [ ]:
import requests
import time

API_KEY = 'YOUR_GOOGLE_MAPS_API_KEY'

# Sample grid points used in the API demonstration.
grid_points = [
    (43.716, 10.396), (43.717, 10.396),
    (43.718, 10.396), (43.719, 10.396)
]

# Collect unique places by place ID.
unique_places = {}

def fetch_places(lat, lon, radius=100):
    all_results = []
    url = (
        f"https://maps.googleapis.com/maps/api/place/nearbysearch/json?"
        f"location={lat},{lon}&radius={radius}&type=tourist_attraction&key={API_KEY}"
    )
    while url:
        resp = requests.get(url)
        data = resp.json()
        all_results.extend(data.get("results", []))
        next_page = data.get("next_page_token")
        if next_page:
            time.sleep(2)
            url = (
                f"https://maps.googleapis.com/maps/api/place/nearbysearch/json?"
                f"pagetoken={next_page}&key={API_KEY}"
            )
        else:
            break
    return all_results

In [ ]:
# Query each grid point.
for lat, lon in grid_points:
    print(f"Querying around: {lat}, {lon}")
    places = fetch_places(lat, lon)
    for place in places:
        pid = place.get("place_id")
        if pid and pid not in unique_places:
            unique_places[pid] = {
                "name": place.get("name"),
                "user_ratings_total": place.get("user_ratings_total"),
                "lat": place["geometry"]["location"]["lat"],
                "lon": place["geometry"]["location"]["lng"]
            }

